# Preparing the YouTube Streaming Lambda Deployment Package

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AccelerationConsortium/streamingLambda/blob/main/streaming_lambda_deployment_prep.ipynb)

This notebook packages the YouTube streaming Lambda function and its dependencies into a `deployment.zip` file that can be uploaded directly to AWS Lambda, per the [guidelines in the AWS docs](https://docs.aws.amazon.com/lambda/latest/dg/python-package.html). This approach does **not** require [AWS Chalice](https://github.com/aws/chalice) and is intended for users outside the Acceleration Consortium who want to set up their own YouTube streaming Lambda function.

See also:
- [AC Training Lab - PiCam Device Setup](https://ac-training-lab.readthedocs.io/en/latest/devices/picam.html)
- [streamingLambda DEPLOYMENT.md](https://github.com/AccelerationConsortium/streamingLambda/blob/main/DEPLOYMENT.md) for full deployment instructions
- [ACC HelloWorld Data Logging](https://github.com/ACC-HelloWorld/5-data-logging) for additional context

This notebook is intended to be run on Google Colab.

## Step 1: Create the dependencies directory

In [ ]:
%mkdir dependencies

## Step 2: Install the required Python dependencies

We install the dependencies from the [requirements.txt](https://github.com/AccelerationConsortium/streamingLambda/blob/main/requirements.txt), **excluding** `chalice` (which is only needed for the Chalice deployment path).

In [ ]:
%pip install --target ./dependencies boto3 google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2

## Step 3: Download the Lambda function and helper modules

Instead of writing the Lambda function code inline, we download `lambda_function.py` and the `chalicelib/` helper module directly from the [streamingLambda repository](https://github.com/AccelerationConsortium/streamingLambda).

In [ ]:
!wget -q https://raw.githubusercontent.com/AccelerationConsortium/streamingLambda/main/lambda_function.py
!echo "Downloaded lambda_function.py"
!cat lambda_function.py

In [ ]:
%mkdir chalicelib
!wget -q -O chalicelib/__init__.py https://raw.githubusercontent.com/AccelerationConsortium/streamingLambda/main/chalicelib/__init__.py
!wget -q -O chalicelib/ytb_api_utils.py https://raw.githubusercontent.com/AccelerationConsortium/streamingLambda/main/chalicelib/ytb_api_utils.py
!echo "Downloaded chalicelib/ contents:"
!ls -la chalicelib/

## Step 4: Create the deployment.zip

First, zip the dependencies directory, then add the Lambda function and `chalicelib/` module to the zip.

In [ ]:
%cd dependencies
!zip -r -q ../deployment.zip .
%cd ..
!zip -q deployment.zip lambda_function.py
!zip -r -q deployment.zip chalicelib/
!echo "deployment.zip created successfully"
!ls -lh deployment.zip

## Step 5: Download the deployment.zip file

In [ ]:
import sys

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('deployment.zip')

NOTE: If the above cell doesn't work in Google Colab, you can download `deployment.zip` using the left navigation bar by **hovering over the file**, clicking the three vertical dots, and clicking "download", as shown below. If you don't see the file, you may need to click the refresh icon near the top of the sidebar. You will then upload this `.zip` file to AWS Lambda.

## Next Steps

After downloading `deployment.zip`, follow the [DEPLOYMENT.md](https://github.com/AccelerationConsortium/streamingLambda/blob/main/DEPLOYMENT.md) guide to:

1. **Create an S3 bucket** for your YouTube API `token.pickle`
2. **Create an IAM role** with the proper permissions for Lambda and S3
3. **Create a Lambda function** in the AWS Console (Python 3.11, 1024 MB memory, 60s timeout)
4. **Upload `deployment.zip`** to the Lambda function
5. **Create a Function URL** to get your `LAMBDA_FUNCTION_URL`
6. **Test the function** using the AWS Console test feature

For the full PiCam device setup, see the [AC Training Lab documentation](https://ac-training-lab.readthedocs.io/en/latest/devices/picam.html).